# Week 23: Next-Gen AI Reasoning, MCTS & Test-Time Compute
## Notebook 06: Synthetic Reasoning Data Generation & Rejection Sampling

### 1. Conceptual Intuition & Real-World Use Case
Human text on the internet is running out, and much of it lacks deep step-by-step reasoning.
To train frontier reasoning models (DeepSeek-R1, LLaMA-3, Nemotron):
- **Synthetic Data Generation Pipelines** programmatically generate millions of reasoning challenges.
- **Rejection Sampling:** Generates $K$ candidate reasoning traces for each problem, validates them with unit tests and compilers, and **keeps only the 100% verified solutions**!
- Used to bootstrap distillation datasets that teach smaller models how to reason!

---

### 2. Mathematical Foundations: Rejection Sampling Filter
Let $G_\theta(x)$ generate $K$ candidate solutions $\{y_1, \dots, y_K\}$ for problem $x$.
Let $V(x, y) \in \{0, 1\}$ be a deterministic verifier (e.g. Python unit test suite):

$$\mathcal{D}_{\text{synthetic}} = \left\{ (x, y_k) \mid V(x, y_k) = 1 \right\}$$
Only solutions passing all unit tests are retained for supervised fine-tuning (SFT).

In [ ]:
import random

# Step 1: Programmatic Generator for Complex Math Word Problems
def generate_math_problem():
    apples = random.randint(5, 20)
    eaten = random.randint(1, 4)
    bought = random.randint(3, 10)
    answer = apples - eaten + bought
    
    question = f"Alice had {apples} apples. She ate {eaten} apples, then bought {bought} more. How many apples does she have now?"
    return question, answer

# Step 2: Simulate LLM Generator with 30% Hallucination Rate
def mock_llm_generator(question, true_ans):
    # Generates a reasoning trace and proposed answer
    if random.random() < 0.70:
        # Correct derivation
        trace = f"Initial: {question.split()[2]}. Ate: subtract. Bought: add. Result = {true_ans}"
        proposed = true_ans
    else:
        # Buggy derivation
        proposed = true_ans + random.choice([-2, 2, 5])
        trace = f"Calculation error. Result = {proposed}"
    return trace, proposed

# Step 3: Implement Rejection Sampling Pipeline
random.seed(42)
golden_dataset = []

total_generated = 10
for i in range(total_generated):
    q, ground_truth = generate_math_problem()
    trace, proposed_ans = mock_llm_generator(q, ground_truth)
    
    # Deterministic Verifier: Check if proposed == ground_truth
    if proposed_ans == ground_truth:
        golden_dataset.append({
            "question": q,
            "reasoning_trace": trace,
            "verified_answer": proposed_ans
        })

print("--- Rejection Sampling Data Pipeline ---")
print(f"Total Candidate Samples Generated: {total_generated}")
print(f"Verified Golden Samples Retained:  {len(golden_dataset)}")
print(f"Rejection Rate (Filtered out):     {(total_generated - len(golden_dataset)) / total_generated * 100:.1f}%")

print("\nCurated Golden Training Sample:")
print("Question:", golden_dataset[0]["question"])
print("Trace:   ", golden_dataset[0]["reasoning_trace"])
print("Answer:  ", golden_dataset[0]["verified_answer"])

### 3. Step-by-Step Code Walkthrough
1. **Programmatic Problem Synthesis:** Creates unambiguous questions with exact ground-truth answers.
2. **Rejection Filtering:** Flawed reasoning traces are automatically discarded.
3. **Golden SFT Curation:** Produces clean training pairs for supervised fine-tuning.

---

### 4. Industry Applications & Trade-offs
- **DeepSeek-R1 & OpenAI o1:** Distilling reasoning traces from large models into small 1.5B/7B models using millions of verified synthetic examples.
- **Trade-off:** Synthetic data can cause mode collapse or domain narrowness if the problem generator lacks diversity.

---

### 5. Top Interview Questions & Answers
**Q1: What is the "Model Collapse" phenomenon when training models recursively on synthetic data?**
> **Answer:** Model collapse occurs when a model is trained on uncurated synthetic outputs of previous models without ground truth anchoring. Over generations, low-probability tails of the true distribution disappear, causing variance loss and degradation into repetitive, meaningless outputs. Rejection sampling with strict verifiers prevents this.

**Q2: How does STaR (Self-Taught Reasoner) bootstrap reasoning?**
> **Answer:** STaR generates reasoning traces for problems. If the answer is correct, the trace is added to the training set. If the answer is incorrect, the model is provided with the correct final answer and asked to "rationalize" backward to find the correct path, doubling training data yield.